# 🔥 Forest Fire Risk RS
## Система аналізу ризику лісових пожеж

Цей ноутбук дозволяє завантажувати супутникові знімки для аналізу лісових пожеж в Україні.

**Можливості:**
- Вибір області з відкритого джерела GeoBoundaries
- Три режими завантаження: вся область, власна зона інтересу, лише території з пожежами
- Автоматичне завантаження порівняльних знімків (попередні місяці та попередній рік)
- Інтеграція з NASA FIRMS та Planet Labs
- Детальне логування (без збереження API ключів)

In [ ]:
# Встановлення залежностей
!pip install -q geopandas requests ipywidgets shapely fiona pyproj rtree ipyfilechooser folium

In [ ]:
import os
import json
import zipfile
import requests
import geopandas as gpd
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from dataclasses import dataclass, field
from io import BytesIO
import uuid
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from ipyfilechooser import FileChooser
import folium
from shapely.geometry import shape, box
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Підключення Google Drive (опційно)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)

## Конфігурація та допоміжні класи

In [ ]:
@dataclass
class ComparisonConfig:
    """Налаштування порівняльних знімків"""
    include_previous_months: bool = True
    previous_months_count: int = 2
    include_previous_year: bool = True
    previous_years_count: int = 1
    
    def generate_periods(self, start_date: datetime, end_date: datetime) -> List[Dict]:
        """Генерує періоди для порівняння"""
        periods = []
        
        # Попередні місяці
        if self.include_previous_months:
            for i in range(1, self.previous_months_count + 1):
                prev_start = start_date - relativedelta(months=i)
                prev_end = end_date - relativedelta(months=i)
                periods.append({
                    'type': 'previous_month',
                    'offset': i,
                    'start': prev_start,
                    'end': prev_end,
                    'description': f'{i} міс. тому',
                    'folder_suffix': f'prev_{i}_months'
                })
        
        # Ті ж місяці в попередньому році
        if self.include_previous_year:
            for i in range(1, self.previous_years_count + 1):
                prev_start = start_date - relativedelta(years=i)
                prev_end = end_date - relativedelta(years=i)
                periods.append({
                    'type': 'previous_year',
                    'offset': i,
                    'start': prev_start,
                    'end': prev_end,
                    'description': f'Ті ж місяці {i} р. тому',
                    'folder_suffix': f'prev_{i}_year'
                })
        
        return periods


@dataclass
class SessionLog:
    """Лог сесії завантаження (без API ключів!)"""
    session_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    started_at: datetime = field(default_factory=datetime.now)
    region_name: str = ""
    download_mode: str = ""
    output_folder: str = ""
    fire_period: Optional[Tuple[datetime, datetime]] = None
    comparison_config: Optional[ComparisonConfig] = None
    aoi_file_name: Optional[str] = None
    downloaded_files: List[Dict] = field(default_factory=list)
    firms_data_file: Optional[str] = None
    boundary_file: Optional[str] = None
    errors: List[str] = field(default_factory=list)
    data_sources: List[str] = field(default_factory=list)
    
    def add_file(self, filename: str, file_type: str, size_bytes: int = 0):
        self.downloaded_files.append({
            'filename': filename,
            'type': file_type,
            'size_bytes': size_bytes,
            'downloaded_at': datetime.now().isoformat()
        })
    
    def add_error(self, error: str):
        self.errors.append(f"[{datetime.now().strftime('%H:%M:%S')}] {error}")
    
    def to_log_string(self) -> str:
        """Генерує текстовий лог (без API ключів!)"""
        lines = [
            "=" * 70,
            "           FOREST FIRE RISK RS - DOWNLOAD LOG",
            "=" * 70,
            "",
            f"Session ID: {self.session_id}",
            f"Started: {self.started_at.strftime('%Y-%m-%d %H:%M:%S')}",
            f"Ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "-" * 70,
            "                    CONFIGURATION",
            "-" * 70,
            "",
            f"Region: {self.region_name}",
            f"Mode: {self.download_mode}",
            f"Output Folder: {self.output_folder}",
        ]
        
        if self.aoi_file_name:
            lines.append(f"AOI File: {self.aoi_file_name}")
        
        if self.fire_period:
            lines.append(f"\nFire Period: {self.fire_period[0].strftime('%d.%m.%Y')} - {self.fire_period[1].strftime('%d.%m.%Y')}")
        
        if self.comparison_config:
            lines.extend([
                "\nComparison Settings:",
                f"  - Previous months: {self.comparison_config.include_previous_months} (count: {self.comparison_config.previous_months_count})",
                f"  - Previous year: {self.comparison_config.include_previous_year} (years: {self.comparison_config.previous_years_count})"
            ])
        
        lines.extend([
            "",
            "-" * 70,
            "                    DATA SOURCES",
            "-" * 70,
            ""
        ])
        
        for source in self.data_sources:
            lines.append(f"• {source}")
        
        lines.extend([
            "",
            "-" * 70,
            "                    DOWNLOADED FILES",
            "-" * 70,
            ""
        ])
        
        if self.firms_data_file:
            lines.append(f"FIRMS Data: {self.firms_data_file}")
        if self.boundary_file:
            lines.append(f"Boundary File: {self.boundary_file}")
        
        lines.append(f"\nTotal Files: {len(self.downloaded_files)}")
        total_size = sum(f.get('size_bytes', 0) for f in self.downloaded_files)
        lines.append(f"Total Size: {total_size / 1024 / 1024:.2f} MB")
        
        for f in self.downloaded_files:
            lines.append(f"\n  • {f['filename']}")
            lines.append(f"    Type: {f['type']} | Size: {f.get('size_bytes', 0) / 1024:.1f} KB")
        
        if self.errors:
            lines.extend([
                "",
                "-" * 70,
                "                    ERRORS",
                "-" * 70,
                ""
            ])
            for err in self.errors:
                lines.append(f"  {err}")
        
        lines.extend([
            "",
            "=" * 70,
            "                    END OF LOG",
            "=" * 70
        ])
        
        return "\n".join(lines)
    
    def save(self, folder: str):
        """Зберігає лог у файл"""
        log_path = os.path.join(folder, 'download_log.txt')
        with open(log_path, 'w', encoding='utf-8') as f:
            f.write(self.to_log_string())
        
        # Також зберігаємо JSON версію (без ключів)
        json_path = os.path.join(folder, 'download_log.json')
        log_dict = {
            'session_id': self.session_id,
            'started_at': self.started_at.isoformat(),
            'ended_at': datetime.now().isoformat(),
            'region_name': self.region_name,
            'download_mode': self.download_mode,
            'output_folder': self.output_folder,
            'downloaded_files': self.downloaded_files,
            'errors': self.errors,
            'data_sources': self.data_sources
        }
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(log_dict, f, indent=2, ensure_ascii=False)

## API Клієнти

In [ ]:
class GeoBoundariesClient:
    """Клієнт для отримання адміністративних меж з GeoBoundaries"""
    
    BASE_URL = "https://www.geoboundaries.org/api/current/gbOpen"
    
    # Українські назви областей
    UA_NAMES = {
        'Vinnytsia': 'Вінницька область',
        'Volyn': 'Волинська область',
        'Dnipropetrovsk': 'Дніпропетровська область',
        'Donetsk': 'Донецька область',
        'Zhytomyr': 'Житомирська область',
        'Zakarpattia': 'Закарпатська область',
        'Zaporizhzhia': 'Запорізька область',
        'Ivano-Frankivsk': 'Івано-Франківська область',
        'Kyiv': 'Київська область',
        'Kirovohrad': 'Кіровоградська область',
        'Luhansk': 'Луганська область',
        'Lviv': 'Львівська область',
        'Mykolaiv': 'Миколаївська область',
        'Odesa': 'Одеська область',
        'Poltava': 'Полтавська область',
        'Rivne': 'Рівненська область',
        'Sumy': 'Сумська область',
        'Ternopil': 'Тернопільська область',
        'Kharkiv': 'Харківська область',
        'Kherson': 'Херсонська область',
        'Khmelnytskyi': 'Хмельницька область',
        'Cherkasy': 'Черкаська область',
        'Chernivtsi': 'Чернівецька область',
        'Chernihiv': 'Чернігівська область',
        'Crimea': 'АР Крим',
        'Sevastopol': 'Севастополь',
    }
    
    @classmethod
    def get_ukraine_oblasts(cls) -> gpd.GeoDataFrame:
        """Отримує всі області України"""
        url = f"{cls.BASE_URL}/UKR/ADM1/"
        
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        # Отримуємо URL для GeoJSON
        geojson_url = data.get('gjDownloadURL') or data.get('simplifiedGeometryGeoJSON')
        if not geojson_url:
            raise ValueError("Не вдалося отримати URL для GeoJSON")
        
        # Завантажуємо GeoJSON
        gdf = gpd.read_file(geojson_url)
        
        # Додаємо українські назви
        def get_ua_name(en_name):
            for key, value in cls.UA_NAMES.items():
                if key.lower() in en_name.lower():
                    return value
            return en_name
        
        gdf['name_ua'] = gdf['shapeName'].apply(get_ua_name)
        gdf['display_name'] = gdf.apply(lambda r: f"{r['name_ua']} ({r['shapeName']})", axis=1)
        
        return gdf.sort_values('name_ua')


class FIRMSClient:
    """
    Клієнт для NASA FIRMS API
    
    Документація: https://firms.modaps.eosdis.nasa.gov/api/
    
    Доступні джерела даних:
    - NRT (Near Real-Time, до 7-10 днів):
        - MODIS_NRT
        - VIIRS_SNPP_NRT
        - VIIRS_NOAA20_NRT
        - VIIRS_NOAA21_NRT
    - SP (Standard Processing, архівні дані):
        - MODIS_SP (з листопада 2000)
        - VIIRS_SNPP_SP (з січня 2012)
        - VIIRS_NOAA20_SP (з квітня 2018)
    """
    
    # API endpoints
    AREA_API_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
    COUNTRY_API_URL = "https://firms.modaps.eosdis.nasa.gov/api/country/csv"
    
    # Доступні джерела даних
    SOURCES = {
        # Near Real-Time (останні 7-10 днів)
        'nrt': {
            'MODIS_NRT': 'MODIS Near Real-Time',
            'VIIRS_SNPP_NRT': 'VIIRS S-NPP Near Real-Time',
            'VIIRS_NOAA20_NRT': 'VIIRS NOAA-20 Near Real-Time',
            'VIIRS_NOAA21_NRT': 'VIIRS NOAA-21 Near Real-Time',
        },
        # Standard Processing (архівні дані)
        'sp': {
            'MODIS_SP': 'MODIS Standard (з 2000)',
            'VIIRS_SNPP_SP': 'VIIRS S-NPP Standard (з 2012)',
            'VIIRS_NOAA20_SP': 'VIIRS NOAA-20 Standard (з 2018)',
        }
    }
    
    def __init__(self, api_key: str):
        self.api_key = api_key
    
    def get_available_sources(self) -> Dict:
        """Повертає список доступних джерел даних"""
        return self.SOURCES
    
    def _determine_source(self, start_date: datetime, end_date: datetime, 
                          preferred_source: str = None) -> Tuple[str, bool]:
        """
        Визначає оптимальне джерело даних на основі дат.
        
        Повертає (source_name, is_archive)
        """
        today = datetime.now()
        days_ago = (today - end_date).days
        
        if preferred_source:
            is_archive = preferred_source.endswith('_SP')
            return preferred_source, is_archive
        
        # Якщо дані за останні 10 днів - використовуємо NRT
        if days_ago <= 10:
            return 'VIIRS_SNPP_NRT', False
        else:
            # Для старших даних використовуємо SP (Standard Processing)
            return 'VIIRS_SNPP_SP', True
    
    def get_fires_by_area(self, bbox: Tuple[float, float, float, float], 
                          start_date: datetime, end_date: datetime,
                          source: str = None) -> gpd.GeoDataFrame:
        """
        Отримує дані про пожежі для заданого bbox та періоду.
        
        Args:
            bbox: (min_lon, min_lat, max_lon, max_lat) - bounding box
            start_date: початкова дата
            end_date: кінцева дата
            source: джерело даних (MODIS_NRT, VIIRS_SNPP_SP, тощо)
        
        Returns:
            GeoDataFrame з точками пожеж
        """
        days = (end_date - start_date).days + 1
        
        # Визначаємо джерело даних
        source, is_archive = self._determine_source(start_date, end_date, source)
        
        print(f"   Джерело даних: {source}")
        print(f"   Тип запиту: {'Архів (SP)' if is_archive else 'Реальний час (NRT)'}")
        
        # Формуємо координати: west,south,east,north
        area = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
        
        all_fires = []
        
        if is_archive:
            # Для архівних даних робимо запити по частинах (макс 10 днів за раз)
            chunk_size = 10
            current_date = start_date
            
            while current_date <= end_date:
                chunk_end = min(current_date + timedelta(days=chunk_size-1), end_date)
                chunk_days = (chunk_end - current_date).days + 1
                
                # URL з датою для архівних даних
                url = f"{self.AREA_API_URL}/{self.api_key}/{source}/{area}/{chunk_days}/{current_date.strftime('%Y-%m-%d')}"
                
                print(f"   Запит: {current_date.strftime('%Y-%m-%d')} - {chunk_end.strftime('%Y-%m-%d')} ({chunk_days} днів)")
                
                try:
                    df = self._fetch_csv(url)
                    if not df.empty:
                        all_fires.append(df)
                        print(f"      ✓ Знайдено {len(df)} точок")
                    else:
                        print(f"      - Пожеж не знайдено")
                except Exception as e:
                    print(f"      ⚠ Помилка: {str(e)}")
                
                current_date = chunk_end + timedelta(days=1)
        else:
            # Для NRT даних - один запит
            url = f"{self.AREA_API_URL}/{self.api_key}/{source}/{area}/{min(days, 10)}"
            
            print(f"   Запит NRT даних за {min(days, 10)} днів")
            
            try:
                df = self._fetch_csv(url)
                if not df.empty:
                    all_fires.append(df)
                    print(f"   ✓ Знайдено {len(df)} точок")
            except Exception as e:
                print(f"   ⚠ Помилка: {str(e)}")
        
        if not all_fires:
            return gpd.GeoDataFrame()
        
        # Об'єднуємо всі результати
        combined_df = pd.concat(all_fires, ignore_index=True)
        
        # Фільтруємо за датою
        if 'acq_date' in combined_df.columns:
            combined_df['acq_date'] = pd.to_datetime(combined_df['acq_date'])
            combined_df = combined_df[
                (combined_df['acq_date'] >= start_date) & 
                (combined_df['acq_date'] <= end_date)
            ]
        
        if combined_df.empty:
            return gpd.GeoDataFrame()
        
        # Створюємо GeoDataFrame
        gdf = gpd.GeoDataFrame(
            combined_df,
            geometry=gpd.points_from_xy(combined_df.longitude, combined_df.latitude),
            crs="EPSG:4326"
        )
        
        return gdf
    
    def get_fires_by_country(self, country_code: str,
                             start_date: datetime, end_date: datetime,
                             source: str = None) -> gpd.GeoDataFrame:
        """
        Отримує дані про пожежі для країни.
        
        Args:
            country_code: 3-літерний код країни (UKR для України)
            start_date: початкова дата
            end_date: кінцева дата
            source: джерело даних
        
        Returns:
            GeoDataFrame з точками пожеж
        """
        days = (end_date - start_date).days + 1
        source, is_archive = self._determine_source(start_date, end_date, source)
        
        print(f"   Джерело даних: {source}")
        print(f"   Країна: {country_code}")
        
        all_fires = []
        
        if is_archive:
            chunk_size = 10
            current_date = start_date
            
            while current_date <= end_date:
                chunk_end = min(current_date + timedelta(days=chunk_size-1), end_date)
                chunk_days = (chunk_end - current_date).days + 1
                
                url = f"{self.COUNTRY_API_URL}/{self.api_key}/{source}/{country_code}/{chunk_days}/{current_date.strftime('%Y-%m-%d')}"
                
                print(f"   Запит: {current_date.strftime('%Y-%m-%d')} - {chunk_end.strftime('%Y-%m-%d')}")
                
                try:
                    df = self._fetch_csv(url)
                    if not df.empty:
                        all_fires.append(df)
                        print(f"      ✓ Знайдено {len(df)} точок")
                except Exception as e:
                    print(f"      ⚠ Помилка: {str(e)}")
                
                current_date = chunk_end + timedelta(days=1)
        else:
            url = f"{self.COUNTRY_API_URL}/{self.api_key}/{source}/{country_code}/{min(days, 10)}"
            
            try:
                df = self._fetch_csv(url)
                if not df.empty:
                    all_fires.append(df)
                    print(f"   ✓ Знайдено {len(df)} точок")
            except Exception as e:
                print(f"   ⚠ Помилка: {str(e)}")
        
        if not all_fires:
            return gpd.GeoDataFrame()
        
        combined_df = pd.concat(all_fires, ignore_index=True)
        
        if 'acq_date' in combined_df.columns:
            combined_df['acq_date'] = pd.to_datetime(combined_df['acq_date'])
            combined_df = combined_df[
                (combined_df['acq_date'] >= start_date) & 
                (combined_df['acq_date'] <= end_date)
            ]
        
        if combined_df.empty:
            return gpd.GeoDataFrame()
        
        gdf = gpd.GeoDataFrame(
            combined_df,
            geometry=gpd.points_from_xy(combined_df.longitude, combined_df.latitude),
            crs="EPSG:4326"
        )
        
        return gdf
    
    def _fetch_csv(self, url: str) -> pd.DataFrame:
        """Завантажує CSV з FIRMS API"""
        response = requests.get(url, timeout=60)
        
        if response.status_code == 401:
            raise ValueError("Невірний API ключ FIRMS. Отримайте ключ на https://firms.modaps.eosdis.nasa.gov/api/")
        elif response.status_code == 400:
            raise ValueError(f"Невірний запит: {response.text}")
        elif response.status_code != 200:
            raise ValueError(f"Помилка API ({response.status_code}): {response.text}")
        
        if not response.text.strip() or response.text.startswith('<!DOCTYPE'):
            return pd.DataFrame()
        
        from io import StringIO
        try:
            df = pd.read_csv(StringIO(response.text))
            return df
        except Exception as e:
            print(f"      Помилка парсингу CSV: {e}")
            return pd.DataFrame()
    
    # Залишаємо старий метод для сумісності
    def get_fires(self, bbox: Tuple[float, float, float, float], 
                  start_date: datetime, end_date: datetime,
                  source: str = None) -> gpd.GeoDataFrame:
        """Alias для get_fires_by_area для сумісності"""
        return self.get_fires_by_area(bbox, start_date, end_date, source)
    
    @staticmethod
    def cluster_fires(fires_gdf: gpd.GeoDataFrame, buffer_km: float = 2.0) -> List[Dict]:
        """Кластеризує точки пожеж"""
        if fires_gdf.empty:
            return []
        
        fires_proj = fires_gdf.to_crs(epsg=3857)
        buffered = fires_proj.buffer(buffer_km * 1000)
        dissolved = buffered.unary_union
        
        clusters = []
        if dissolved.geom_type == 'Polygon':
            geoms = [dissolved]
        elif dissolved.geom_type == 'MultiPolygon':
            geoms = list(dissolved.geoms)
        else:
            return []
        
        for i, geom in enumerate(geoms):
            cluster_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:3857").to_crs("EPSG:4326")
            bounds = cluster_gdf.total_bounds
            centroid = cluster_gdf.centroid.iloc[0]
            
            cluster_fires = fires_gdf[fires_gdf.within(cluster_gdf.iloc[0].geometry)]
            
            clusters.append({
                'id': f'cluster_{i+1}',
                'bbox': tuple(bounds),
                'centroid': (centroid.x, centroid.y),
                'point_count': len(cluster_fires),
                'earliest_date': cluster_fires['acq_date'].min() if not cluster_fires.empty and 'acq_date' in cluster_fires.columns else None,
                'latest_date': cluster_fires['acq_date'].max() if not cluster_fires.empty and 'acq_date' in cluster_fires.columns else None,
                'geometry': cluster_gdf.iloc[0].geometry
            })
        
        return clusters


class PlanetClient:
    """Клієнт для Planet Labs API"""
    
    BASE_URL = "https://api.planet.com/data/v1"
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.session = requests.Session()
        self.session.auth = (api_key, '')
    
    def search_imagery(self, bbox: Tuple[float, float, float, float],
                       start_date: datetime, end_date: datetime,
                       item_types: List[str] = ["PSScene"],
                       max_cloud_cover: float = 20) -> List[Dict]:
        """Пошук знімків"""
        
        geometry = {
            "type": "Polygon",
            "coordinates": [[
                [bbox[0], bbox[1]],
                [bbox[2], bbox[1]],
                [bbox[2], bbox[3]],
                [bbox[0], bbox[3]],
                [bbox[0], bbox[1]]
            ]]
        }
        
        filter_config = {
            "type": "AndFilter",
            "config": [
                {
                    "type": "GeometryFilter",
                    "field_name": "geometry",
                    "config": geometry
                },
                {
                    "type": "DateRangeFilter",
                    "field_name": "acquired",
                    "config": {
                        "gte": start_date.strftime("%Y-%m-%dT00:00:00Z"),
                        "lte": end_date.strftime("%Y-%m-%dT23:59:59Z")
                    }
                },
                {
                    "type": "RangeFilter",
                    "field_name": "cloud_cover",
                    "config": {
                        "lte": max_cloud_cover / 100.0
                    }
                }
            ]
        }
        
        search_request = {
            "item_types": item_types,
            "filter": filter_config
        }
        
        response = self.session.post(
            f"{self.BASE_URL}/quick-search",
            json=search_request
        )
        response.raise_for_status()
        
        return response.json().get('features', [])
    
    def validate_key(self) -> bool:
        """Перевіряє валідність API ключа"""
        try:
            response = self.session.get(f"{self.BASE_URL}/item-types")
            return response.status_code == 200
        except:
            return False

## Допоміжні функції для роботи з файлами

In [ ]:
def parse_uploaded_file(file_path: str) -> gpd.GeoDataFrame:
    """Парсить завантажений файл (GeoJSON, Shapefile, KML, KMZ)"""
    ext = os.path.splitext(file_path)[1].lower()
    
    if ext in ['.geojson', '.json']:
        return gpd.read_file(file_path)
    elif ext == '.shp':
        return gpd.read_file(file_path)
    elif ext == '.kml':
        gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
        return gpd.read_file(file_path, driver='KML')
    elif ext == '.kmz':
        # KMZ - це ZIP з KML всередині
        with zipfile.ZipFile(file_path, 'r') as z:
            for name in z.namelist():
                if name.endswith('.kml'):
                    kml_content = z.read(name)
                    gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
                    return gpd.read_file(BytesIO(kml_content), driver='KML')
        raise ValueError("KML файл не знайдено в KMZ архіві")
    else:
        raise ValueError(f"Непідтримуваний формат файлу: {ext}")


def create_folder_structure(base_path: str, region_name: str, mode: str,
                           fire_period: Optional[Tuple[datetime, datetime]] = None,
                           comparison_config: Optional[ComparisonConfig] = None) -> Dict[str, str]:
    """Створює структуру папок для завантаження"""
    
    # Санітизуємо назву
    safe_name = region_name.replace(' ', '_').replace('/', '_').replace('\\', '_')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    if fire_period:
        folder_name = f"{safe_name}_{mode}_{fire_period[0].strftime('%Y%m%d')}_to_{fire_period[1].strftime('%Y%m%d')}_{timestamp}"
    else:
        folder_name = f"{safe_name}_{mode}_{timestamp}"
    
    session_folder = os.path.join(base_path, folder_name)
    os.makedirs(session_folder, exist_ok=True)
    
    folders = {
        'session': session_folder,
        'data': os.path.join(session_folder, 'data'),
        'logs': os.path.join(session_folder, 'logs')
    }
    
    # Створюємо підпапки
    for folder in folders.values():
        os.makedirs(folder, exist_ok=True)
    
    # Для режиму пожеж - додаткові папки
    if mode == 'fire_areas' and fire_period and comparison_config:
        folders['fire_period'] = os.path.join(
            session_folder,
            f"fire_period_{fire_period[0].strftime('%Y%m%d')}_to_{fire_period[1].strftime('%Y%m%d')}"
        )
        os.makedirs(folders['fire_period'], exist_ok=True)
        
        # Папки для порівняльних періодів
        for period in comparison_config.generate_periods(fire_period[0], fire_period[1]):
            period_folder = os.path.join(
                session_folder,
                f"comparison_{period['folder_suffix']}_{period['start'].strftime('%Y%m%d')}_to_{period['end'].strftime('%Y%m%d')}"
            )
            os.makedirs(period_folder, exist_ok=True)
            folders[f"comparison_{period['folder_suffix']}"] = period_folder
    
    return folders


def save_geodataframe(gdf: gpd.GeoDataFrame, folder: str, filename: str) -> str:
    """Зберігає GeoDataFrame у файл"""
    filepath = os.path.join(folder, filename)
    gdf.to_file(filepath, driver='GeoJSON')
    return filepath

## Головний інтерфейс

In [ ]:
class ForestFireUI:
    """Головний клас інтерфейсу користувача"""
    
    def __init__(self):
        self.oblasts_gdf = None
        self.selected_oblast = None
        self.aoi_gdf = None
        self.log = None
        
        self._create_widgets()
        self._load_oblasts()
    
    def _create_widgets(self):
        """Створює всі віджети"""
        
        # Стилі
        style = {'description_width': '150px'}
        layout = widgets.Layout(width='500px')
        
        # === 1. Вибір області ===
        self.oblast_dropdown = widgets.Dropdown(
            options=[('Завантаження...', None)],
            description='Область:',
            style=style,
            layout=layout
        )
        
        # === 2. Режим завантаження ===
        self.mode_radio = widgets.RadioButtons(
            options=[
                ('Вся область - Завантажити всю територію', 'full_oblast'),
                ('Зона інтересу - Завантажити власну ділянку', 'aoi'),
                ('Лише пожежі - Завантажити тільки території з пожежами', 'fire_areas')
            ],
            value='fire_areas',
            description='Режим:',
            style=style,
            layout=widgets.Layout(width='600px')
        )
        self.mode_radio.observe(self._on_mode_change, names='value')
        
        # === 3. Опції для режиму пожеж ===
        today = datetime.now()
        month_ago = today - timedelta(days=30)
        
        self.fire_start_date = widgets.DatePicker(
            description='Початок:',
            value=month_ago.date(),
            style=style
        )
        
        self.fire_end_date = widgets.DatePicker(
            description='Кінець:',
            value=today.date(),
            style=style
        )
        
        self.include_prev_months = widgets.Checkbox(
            value=True,
            description='Включити попередні місяці',
            style=style
        )
        
        self.prev_months_count = widgets.IntSlider(
            value=2,
            min=1,
            max=6,
            description='Кількість місяців:',
            style=style
        )
        
        self.include_prev_year = widgets.Checkbox(
            value=True,
            description='Ті ж місяці в попередньому році',
            style=style
        )
        
        self.prev_years_count = widgets.IntSlider(
            value=1,
            min=1,
            max=3,
            description='Кількість років:',
            style=style
        )
        
        self.fire_options_box = widgets.VBox([
            widgets.HTML('<h4>Налаштування виявлення пожеж:</h4>'),
            widgets.HBox([self.fire_start_date, self.fire_end_date]),
            widgets.HTML('<br><b>Порівняльні знімки:</b>'),
            self.include_prev_months,
            self.prev_months_count,
            self.include_prev_year,
            self.prev_years_count
        ])
        
        # === 4. Опції для AOI ===
        self.file_upload = widgets.FileUpload(
            accept='.geojson,.json,.shp,.kml,.kmz,.zip',
            multiple=False,
            description='Завантажити файл'
        )
        self.file_upload.observe(self._on_file_upload, names='value')
        
        self.aoi_status = widgets.HTML('<i>Файл не завантажено</i>')
        
        self.aoi_options_box = widgets.VBox([
            widgets.HTML('<h4>Завантажте файл з межами:</h4>'),
            self.file_upload,
            self.aoi_status,
            widgets.HTML('<small>Підтримувані формати: GeoJSON, Shapefile, KML, KMZ</small>')
        ])
        self.aoi_options_box.layout.display = 'none'
        
        # === 5. Опції для всієї області ===
        current_year = datetime.now().year
        
        self.years_select = widgets.SelectMultiple(
            options=list(range(current_year, 2019, -1)),
            value=[current_year],
            description='Роки:',
            style=style,
            rows=6
        )
        
        months = [
            ('01 - Січень', 1), ('02 - Лютий', 2), ('03 - Березень', 3),
            ('04 - Квітень', 4), ('05 - Травень', 5), ('06 - Червень', 6),
            ('07 - Липень', 7), ('08 - Серпень', 8), ('09 - Вересень', 9),
            ('10 - Жовтень', 10), ('11 - Листопад', 11), ('12 - Грудень', 12)
        ]
        self.months_select = widgets.SelectMultiple(
            options=months,
            value=[datetime.now().month],
            description='Місяці:',
            style=style,
            rows=12
        )
        
        self.full_oblast_options_box = widgets.VBox([
            widgets.HTML('<h4>Оберіть періоди для завантаження:</h4>'),
            widgets.HBox([self.years_select, self.months_select])
        ])
        self.full_oblast_options_box.layout.display = 'none'
        
        # === 6. API ключі ===
        self.planet_key = widgets.Password(
            description='Planet API Key:',
            placeholder='Введіть ключ',
            style=style,
            layout=layout
        )
        
        self.firms_key = widgets.Password(
            description='FIRMS MAP Key:',
            placeholder='Обов\'язково для режиму пожеж',
            style=style,
            layout=layout
        )
        
        # === 7. Вихідна папка ===
        self.output_folder = widgets.Text(
            description='Папка виводу:',
            value='/content/drive/MyDrive/ForestFireData' if IN_COLAB else './output',
            style=style,
            layout=layout
        )
        
        # === 8. Кнопка запуску та статус ===
        self.start_button = widgets.Button(
            description='🚀 Почати завантаження',
            button_style='success',
            layout=widgets.Layout(width='200px', height='40px')
        )
        self.start_button.on_click(self._on_start_click)
        
        self.progress = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description='Прогрес:',
            style=style,
            layout=widgets.Layout(width='400px')
        )
        self.progress.layout.display = 'none'
        
        self.status_output = widgets.Output()
    
    def _load_oblasts(self):
        """Завантажує список областей"""
        try:
            self.oblasts_gdf = GeoBoundariesClient.get_ukraine_oblasts()
            options = [(row['display_name'], idx) for idx, row in self.oblasts_gdf.iterrows()]
            self.oblast_dropdown.options = [('Оберіть область...', None)] + options
        except Exception as e:
            self.oblast_dropdown.options = [(f'Помилка: {str(e)}', None)]
    
    def _on_mode_change(self, change):
        """Обробник зміни режиму"""
        mode = change['new']
        
        self.fire_options_box.layout.display = 'block' if mode == 'fire_areas' else 'none'
        self.aoi_options_box.layout.display = 'block' if mode == 'aoi' else 'none'
        self.full_oblast_options_box.layout.display = 'block' if mode == 'full_oblast' else 'none'
    
    def _on_file_upload(self, change):
        """Обробник завантаження файлу"""
        if change['new']:
            try:
                uploaded = list(change['new'].values())[0]
                filename = list(change['new'].keys())[0]
                
                # Зберігаємо тимчасовий файл
                temp_path = f'/tmp/{filename}'
                with open(temp_path, 'wb') as f:
                    f.write(uploaded['content'])
                
                self.aoi_gdf = parse_uploaded_file(temp_path)
                bounds = self.aoi_gdf.total_bounds
                
                self.aoi_status.value = f'''<span style="color: green;">✓ {filename}</span><br>
                <small>Bbox: ({bounds[0]:.4f}, {bounds[1]:.4f}) - ({bounds[2]:.4f}, {bounds[3]:.4f})</small>'''
                
            except Exception as e:
                self.aoi_status.value = f'<span style="color: red;">Помилка: {str(e)}</span>'
                self.aoi_gdf = None
    
    def _on_start_click(self, button):
        """Обробник натискання кнопки старт"""
        with self.status_output:
            clear_output()
            
            # Валідація
            errors = []
            
            if self.oblast_dropdown.value is None:
                errors.append('Оберіть область')
            
            mode = self.mode_radio.value
            
            if mode == 'aoi' and self.aoi_gdf is None:
                errors.append('Завантажте файл з межами')
            
            if mode == 'fire_areas' and not self.firms_key.value:
                errors.append('Введіть FIRMS API ключ')
            
            if mode == 'full_oblast':
                if not self.years_select.value:
                    errors.append('Оберіть хоча б один рік')
                if not self.months_select.value:
                    errors.append('Оберіть хоча б один місяць')
            
            if errors:
                for err in errors:
                    print(f'⚠️ {err}')
                return
            
            # Запуск завантаження
            self._run_download()
    
    def _run_download(self):
        """Виконує завантаження"""
        mode = self.mode_radio.value
        oblast_idx = self.oblast_dropdown.value
        oblast = self.oblasts_gdf.loc[oblast_idx]
        
        self.progress.value = 0
        self.progress.layout.display = 'block'
        
        # Створюємо лог
        self.log = SessionLog(
            region_name=oblast['display_name'],
            download_mode=mode
        )
        self.log.data_sources.append('GeoBoundaries (https://www.geoboundaries.org)')
        
        try:
            if mode == 'fire_areas':
                self._download_fire_areas(oblast)
            elif mode == 'full_oblast':
                self._download_full_oblast(oblast)
            elif mode == 'aoi':
                self._download_aoi(oblast)
            
            print('\n✅ Завантаження завершено!')
            print(f'📁 Файли збережено в: {self.log.output_folder}')
            
        except Exception as e:
            self.log.add_error(str(e))
            print(f'\n❌ Помилка: {str(e)}')
        
        finally:
            # Зберігаємо лог
            if self.log.output_folder:
                logs_folder = os.path.join(self.log.output_folder, 'logs')
                os.makedirs(logs_folder, exist_ok=True)
                self.log.save(logs_folder)
                print(f'📝 Лог збережено в: {logs_folder}')
            
            self.progress.value = 100
    
    def _download_fire_areas(self, oblast):
        """Завантаження для режиму пожеж"""
        print('🔥 Режим: Лише території з пожежами')
        
        start_date = datetime.combine(self.fire_start_date.value, datetime.min.time())
        end_date = datetime.combine(self.fire_end_date.value, datetime.max.time())
        
        comparison_config = ComparisonConfig(
            include_previous_months=self.include_prev_months.value,
            previous_months_count=self.prev_months_count.value,
            include_previous_year=self.include_prev_year.value,
            previous_years_count=self.prev_years_count.value
        )
        
        self.log.fire_period = (start_date, end_date)
        self.log.comparison_config = comparison_config
        self.log.data_sources.append('NASA FIRMS (https://firms.modaps.eosdis.nasa.gov)')
        
        # Створюємо папки
        folders = create_folder_structure(
            self.output_folder.value,
            oblast['shapeName'],
            'fire_areas',
            (start_date, end_date),
            comparison_config
        )
        self.log.output_folder = folders['session']
        
        self.progress.value = 10
        
        # Зберігаємо межі області
        oblast_gdf = gpd.GeoDataFrame([oblast], crs="EPSG:4326")
        boundary_path = save_geodataframe(oblast_gdf, folders['data'], 'oblast_boundary.geojson')
        self.log.boundary_file = boundary_path
        print(f'📍 Збережено межі області: {boundary_path}')
        
        self.progress.value = 20
        
        # Отримуємо пожежі з FIRMS
        bbox = oblast.geometry.bounds
        firms_client = FIRMSClient(self.firms_key.value)
        
        print(f'🔍 Пошук пожеж за період {start_date.strftime("%d.%m.%Y")} - {end_date.strftime("%d.%m.%Y")}...')
        fires_gdf = firms_client.get_fires(bbox, start_date, end_date)
        
        if fires_gdf.empty:
            print('⚠️ Пожеж не знайдено за вказаний період')
            return
        
        # Фільтруємо пожежі, що в межах області
        fires_in_oblast = fires_gdf[fires_gdf.within(oblast.geometry)]
        print(f'🔥 Знайдено {len(fires_in_oblast)} точок пожеж в межах області')
        
        # Зберігаємо дані FIRMS
        firms_path = save_geodataframe(fires_in_oblast, folders['data'], 'firms_fire_data.geojson')
        self.log.firms_data_file = firms_path
        self.log.add_file('firms_fire_data.geojson', 'fire_data', os.path.getsize(firms_path))
        print(f'💾 Збережено дані FIRMS: {firms_path}')
        
        self.progress.value = 40
        
        # Кластеризуємо пожежі
        clusters = FIRMSClient.cluster_fires(fires_in_oblast, buffer_km=2.0)
        print(f'📊 Створено {len(clusters)} кластерів пожеж')
        
        # Показуємо періоди для порівняння
        periods = comparison_config.generate_periods(start_date, end_date)
        if periods:
            print('\n📅 Періоди для порівняння:')
            for p in periods:
                print(f"   • {p['description']}: {p['start'].strftime('%d.%m.%Y')} - {p['end'].strftime('%d.%m.%Y')}")
        
        self.progress.value = 60
        
        # Тут буде завантаження знімків з Planet (якщо ключ є)
        if self.planet_key.value:
            self.log.data_sources.append('Planet Labs (https://api.planet.com)')
            print('\n🛰️ Пошук супутникових знімків Planet...')
            # TODO: Реалізувати завантаження з Planet
            print('   (Функціонал в розробці)')
        
        self.progress.value = 100
    
    def _download_full_oblast(self, oblast):
        """Завантаження для всієї області"""
        print('🗺️ Режим: Вся область')
        
        years = list(self.years_select.value)
        months = list(self.months_select.value)
        
        # Створюємо папки
        folders = create_folder_structure(
            self.output_folder.value,
            oblast['shapeName'],
            'full_oblast'
        )
        self.log.output_folder = folders['session']
        
        # Зберігаємо межі області
        oblast_gdf = gpd.GeoDataFrame([oblast], crs="EPSG:4326")
        boundary_path = save_geodataframe(oblast_gdf, folders['data'], 'oblast_boundary.geojson')
        self.log.boundary_file = boundary_path
        print(f'📍 Збережено межі області: {boundary_path}')
        
        print(f'\n📅 Обрані періоди:')
        for year in years:
            for month in months:
                print(f'   • {year}/{month:02d}')
                # Створюємо папку для періоду
                period_folder = os.path.join(folders['session'], f'{year}_{month:02d}')
                os.makedirs(period_folder, exist_ok=True)
        
        if self.planet_key.value:
            self.log.data_sources.append('Planet Labs (https://api.planet.com)')
            print('\n🛰️ Пошук супутникових знімків Planet...')
            print('   (Функціонал в розробці)')
        
        self.progress.value = 100
    
    def _download_aoi(self, oblast):
        """Завантаження для зони інтересу"""
        print('📍 Режим: Зона інтересу')
        
        self.log.aoi_file_name = 'uploaded_aoi'
        
        # Створюємо папки
        folders = create_folder_structure(
            self.output_folder.value,
            'AOI',
            'aoi'
        )
        self.log.output_folder = folders['session']
        
        # Зберігаємо AOI
        aoi_path = save_geodataframe(self.aoi_gdf, folders['data'], 'aoi_boundary.geojson')
        self.log.boundary_file = aoi_path
        print(f'📍 Збережено межі AOI: {aoi_path}')
        
        bounds = self.aoi_gdf.total_bounds
        print(f'   Bbox: ({bounds[0]:.4f}, {bounds[1]:.4f}) - ({bounds[2]:.4f}, {bounds[3]:.4f})')
        
        if self.planet_key.value:
            self.log.data_sources.append('Planet Labs (https://api.planet.com)')
            print('\n🛰️ Пошук супутникових знімків Planet...')
            print('   (Функціонал в розробці)')
        
        self.progress.value = 100
    
    def display(self):
        """Відображає інтерфейс"""
        
        ui = widgets.VBox([
            widgets.HTML('<h2>🔥 Forest Fire Risk RS</h2>'),
            widgets.HTML('<p>Система аналізу ризику лісових пожеж</p>'),
            widgets.HTML('<hr>'),
            
            widgets.HTML('<h3>1. Оберіть область:</h3>'),
            self.oblast_dropdown,
            
            widgets.HTML('<h3>2. Оберіть режим завантаження:</h3>'),
            self.mode_radio,
            
            self.fire_options_box,
            self.aoi_options_box,
            self.full_oblast_options_box,
            
            widgets.HTML('<h3>3. API ключі:</h3>'),
            self.planet_key,
            self.firms_key,
            
            widgets.HTML('<h3>4. Вихідна папка:</h3>'),
            self.output_folder,
            
            widgets.HTML('<hr>'),
            self.start_button,
            self.progress,
            self.status_output
        ])
        
        display(ui)
        
        # Показуємо опції для режиму за замовчуванням
        self._on_mode_change({'new': self.mode_radio.value})

## Запуск інтерфейсу

In [ ]:
# Створюємо та відображаємо інтерфейс
ui = ForestFireUI()
ui.display()